# 🚀 GCC Scraper — Colab 协作工作台

[![GitHub](https://img.shields.io/badge/GitHub-Source-black?logo=github)](https://github.com/Jungle225TT/GCC-)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-blue)]()
[![Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

---

### 📋 使用流程
```
① 拉取代码  →  ② 挂载 Drive  →  ③ 安装依赖  →  ④ 配置密钥
     ↓
⑤ 选择运行模式（爬虫 / 去重 / 飞书同步）  →  ⑥ 查看结果
```

| 步骤 | 说明 | 预计耗时 |
|------|------|---------|
| 1–4  | 环境初始化（首次运行）| ~3 min |
| 5    | 爬虫抓取 | 视数据量而定 |
| 5    | 去重 | <1 min |
| 5    | 飞书同步 | <2 min |

> **💡 协作说明**：所有同事将 `data/` 目录映射到同一个 Google Drive 共享文件夹，即可共享数据库与输出文件，无需互相传文件。

---
## 📦 第一步：从 GitHub 拉取代码

In [3]:
# @title ⚙️ 配置仓库地址（点击左侧 ▶ 运行）
# @markdown 填入你的 GitHub 仓库地址（支持 HTTPS / SSH）
GITHUB_REPO = "https://github.com/Jungle225TT/GCC-.git"  # @param {type:"string"}
BRANCH = "main"  # @param ["main", "dev", "staging"]
PROJECT_DIR = "/content/gcc_scraper"  # @param {type:"string"}

import os, subprocess

def run(cmd, check=True):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"命令失败: {cmd}")
    return result

if os.path.exists(PROJECT_DIR):
    print("🔄 仓库已存在，执行 git pull 更新...")
    run(f"cd {PROJECT_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}")
else:
    print(f"📥 克隆仓库中: {GITHUB_REPO}")
    run(f"git clone -b {BRANCH} {GITHUB_REPO} {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print(f"\n✅ 当前目录: {os.getcwd()}")
run("ls -la")

📥 克隆仓库中: https://github.com/Jungle225TT/GCC-.git
Cloning into '/content/gcc_scraper'...


✅ 当前目录: /content/gcc_scraper
total 228
drwxr-xr-x 7 root root  4096 Apr 27 12:54 .
drwxr-xr-x 1 root root  4096 Apr 27 12:54 ..
-rw-r--r-- 1 root root  4540 Apr 27 12:54 ai_client.py
drwxr-xr-x 2 root root  4096 Apr 27 12:54 assets
lrwxrwxrwx 1 root root    39 Apr 27 12:54 data -> /content/drive/MyDrive/gcc_project/data
-rw-r--r-- 1 root root  2365 Apr 27 12:54 dedup.py
drwxr-xr-x 2 root root  4096 Apr 27 12:54 docs
-rw-r--r-- 1 root root 13055 Apr 27 12:54 feishu_sync.py
-rw-r--r-- 1 root root  6876 Apr 27 12:54 filter_rules.yaml
-rw-r--r-- 1 root root 30519 Apr 27 12:54 fulltext_to_pdf.py
-rw-r--r-- 1 root root 83041 Apr 27 12:54 gcc_thinktank_scraper_v2.py
drwxr-xr-x 8 root root  4096 Apr 27 12:54 .git
-rw-r--r-- 1 root root  1526 Apr 27 12:54 .gitignore
drwxr-xr-x 2 root root  4096 Apr 27 12:54 notebooks
lrwxrwxrwx 1 root root    41 Apr 27 12:54 output -> /content/drive/MyDrive/gcc_project/out

CompletedProcess(args='ls -la', returncode=0, stdout='total 228\ndrwxr-xr-x 7 root root  4096 Apr 27 12:54 .\ndrwxr-xr-x 1 root root  4096 Apr 27 12:54 ..\n-rw-r--r-- 1 root root  4540 Apr 27 12:54 ai_client.py\ndrwxr-xr-x 2 root root  4096 Apr 27 12:54 assets\nlrwxrwxrwx 1 root root    39 Apr 27 12:54 data -> /content/drive/MyDrive/gcc_project/data\n-rw-r--r-- 1 root root  2365 Apr 27 12:54 dedup.py\ndrwxr-xr-x 2 root root  4096 Apr 27 12:54 docs\n-rw-r--r-- 1 root root 13055 Apr 27 12:54 feishu_sync.py\n-rw-r--r-- 1 root root  6876 Apr 27 12:54 filter_rules.yaml\n-rw-r--r-- 1 root root 30519 Apr 27 12:54 fulltext_to_pdf.py\n-rw-r--r-- 1 root root 83041 Apr 27 12:54 gcc_thinktank_scraper_v2.py\ndrwxr-xr-x 8 root root  4096 Apr 27 12:54 .git\n-rw-r--r-- 1 root root  1526 Apr 27 12:54 .gitignore\ndrwxr-xr-x 2 root root  4096 Apr 27 12:54 notebooks\nlrwxrwxrwx 1 root root    41 Apr 27 12:54 output -> /content/drive/MyDrive/gcc_project/output\nlrwxrwxrwx 1 root root    50 Apr 27 12:54 out

In [4]:
# @title 🔐 私有仓库认证（仅私有仓库需要运行此格）
# @markdown 推荐使用 GitHub Personal Access Token（Settings → Developer settings → Tokens）
# @markdown **不要直接填入密码，使用 Colab Secrets 更安全**

USE_PAT = False  # @param {type:"boolean"}

if USE_PAT:
    from google.colab import userdata
    try:
        token = userdata.get('GITHUB_TOKEN')
        repo_url = GITHUB_REPO.replace("https://", f"https://{token}@")
        run(f"git remote set-url origin {repo_url}")
        print("✅ 已配置 PAT 认证")
    except Exception as e:
        print(f"⚠️ 请先在 Colab Secrets 中添加 GITHUB_TOKEN: {e}")
        print("路径：左侧工具栏 🔑 图标 → 添加密钥")

---
## 💾 第二步：挂载 Google Drive（共享数据目录）

In [5]:
# @title 📂 挂载 Drive 并链接输出目录
# @markdown **协作使用**：所有同事将下方路径指向同一个 Drive 共享文件夹，即可共享数据库和抓取结果
DRIVE_ROOT = "MyDrive/gcc_project"  # @param {type:"string"}
# @markdown `DRIVE_ROOT` 是 Drive 里的项目根目录，以下子目录会自动创建：
# @markdown - `data/` → gcc_dedup.db 数据库
# @markdown - `output/` → 爬虫抓取结果（JSON/CSV）
# @markdown - `output_fulltext/` → 全文 PDF 输出

import os, shutil
from google.colab import drive

# ── 兜底：若第一步未运行或内核重启，自动恢复 PROJECT_DIR ──
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/gcc_scraper")
if not os.path.exists(PROJECT_DIR):
    raise RuntimeError(f"❌ 项目目录不存在: {PROJECT_DIR}\n   请先运行第一步拉取代码！")

drive.mount('/content/drive', force_remount=False)

SYNC_DIRS = {
    "data":            "data",
    "output":          "output",
    "output_fulltext": "output_fulltext",
}

print("🔗 建立 Drive 符号链接：")
for local_name, drive_subdir in SYNC_DIRS.items():
    local_path = f"{PROJECT_DIR}/{local_name}"
    drive_path = f"/content/drive/{DRIVE_ROOT}/{drive_subdir}"
    os.makedirs(drive_path, exist_ok=True)

    if os.path.islink(local_path):
        os.unlink(local_path)
    elif os.path.isdir(local_path):
        for fname in os.listdir(local_path):
            src = f"{local_path}/{fname}"
            dst = f"{drive_path}/{fname}"
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
        shutil.rmtree(local_path)

    os.symlink(drive_path, local_path)
    print(f"  ✅ {local_name}/  →  {drive_path}")

DRIVE_DATA_PATH = f"{DRIVE_ROOT}/data"
print(f"\n✅ Drive 挂载完成！所有输出文件自动同步到 Drive")
print(f"📂 Drive 项目目录: /content/drive/{DRIVE_ROOT}")


Mounted at /content/drive
🔗 建立 Drive 符号链接：
  ✅ data/  →  /content/drive/MyDrive/gcc_project/data
  ✅ output/  →  /content/drive/MyDrive/gcc_project/output
  ✅ output_fulltext/  →  /content/drive/MyDrive/gcc_project/output_fulltext

✅ Drive 挂载完成！所有输出文件自动同步到 Drive
📂 Drive 项目目录: /content/drive/MyDrive/gcc_project


---
## 🛠️ 第三步：安装依赖

In [6]:
# @title 📦 安装 requirements.txt（带缓存加速）
# @markdown 首次运行约 2–3 分钟；Drive 缓存后后续运行 <30 秒
import os
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/gcc_scraper")

USE_DRIVE_CACHE = True  # @param {type:"boolean"}

import subprocess, sys, os, hashlib

req_file = f"{PROJECT_DIR}/requirements.txt"
cache_dir = f"/content/drive/{DRIVE_DATA_PATH}/../pip_cache"

if USE_DRIVE_CACHE:
    os.makedirs(cache_dir, exist_ok=True)
    pip_cmd = f"pip install -r {req_file} --cache-dir {cache_dir} -q"
else:
    pip_cmd = f"pip install -r {req_file} -q"

print("📦 安装中，请稍候...")
result = subprocess.run(pip_cmd, shell=True, capture_output=True, text=True)
if result.returncode == 0:
    print("✅ 依赖安装完成！")
else:
    print("⚠️ 安装遇到问题：")
    print(result.stderr[-2000:])

# 验证关键包
critical = ["requests", "pandas", "sqlite3"]
print("\n🔍 关键包验证：")
for pkg in critical:
    try:
        __import__(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg} — 请检查 requirements.txt")

📦 安装中，请稍候...
✅ 依赖安装完成！

🔍 关键包验证：
  ✅ requests
  ✅ pandas
  ✅ sqlite3


---
## 🔑 第四步：配置 API 密钥

> **安全建议**：密钥统一存放在 Colab Secrets（左侧 🔑 图标），**不要**硬编码在代码里。
>
> 需要配置的密钥：
>
> | Secrets 键名 | 说明 | 是否必填 |
> |---|---|---|
> | `DEEPSEEK_API_KEY` | DeepSeek API Key（默认 AI）| ✅ 必填 |
> | `FEISHU_APP_ID` | 飞书应用 App ID | 飞书同步时必填 |
> | `FEISHU_APP_SECRET` | 飞书应用 App Secret | 飞书同步时必填 |
> | `AI_PROVIDER` | 填 `anthropic` 可切换到 Claude | 可选 |
> | `ANTHROPIC_API_KEY` | Claude API Key | 切换 Claude 时填 |
> | `GITHUB_TOKEN` | GitHub PAT（私有仓库）| 可选 |
>
> **添加方法**：左侧工具栏 → 🔑 图标 → `+ Add new secret` → 填入 Name 和 Value → 开启右侧 Notebook access 开关

In [7]:
# @title 🔐 加载密钥到环境变量
import os
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/gcc_scraper")

from google.colab import userdata

# 密钥配置：(Secrets键名, 环境变量名, 是否必填)
SECRET_CONFIGS = [
    ("DEEPSEEK_API_KEY",   "DEEPSEEK_API_KEY",   True),
    ("FEISHU_APP_ID",      "FEISHU_APP_ID",      False),
    ("FEISHU_APP_SECRET",  "FEISHU_APP_SECRET",  False),
    ("AI_PROVIDER",        "AI_PROVIDER",        False),
    ("ANTHROPIC_API_KEY",  "ANTHROPIC_API_KEY",  False),
]

loaded, missing_required, missing_optional = [], [], []
for secret_name, env_name, required in SECRET_CONFIGS:
    try:
        val = userdata.get(secret_name)
        os.environ[env_name] = val
        loaded.append(secret_name)
    except Exception:
        if required:
            missing_required.append(secret_name)
        else:
            missing_optional.append(secret_name)

print("🔑 密钥加载状态：")
for k in loaded:            print(f"  ✅ {k}")
for k in missing_required:  print(f"  ❌ {k}  ← 必填，AI 功能将不可用")
for k in missing_optional:  print(f"  ⚪ {k}  （可选，暂未配置）")

# 显示当前 AI Provider
provider = os.environ.get("AI_PROVIDER", "deepseek")
print(f"\n🤖 当前 AI Provider: {provider.upper()}")
if provider == "deepseek":
    has_key = bool(os.environ.get("DEEPSEEK_API_KEY"))
    print(f"   DEEPSEEK_API_KEY: {'✅ 已加载' if has_key else '❌ 未配置'}")
else:
    has_key = bool(os.environ.get("ANTHROPIC_API_KEY"))
    print(f"   ANTHROPIC_API_KEY: {'✅ 已加载' if has_key else '❌ 未配置'}")

if missing_required:
    print("\n📌 添加方法：左侧工具栏 → 🔑 图标 → + Add new secret → 开启 Notebook access")


🔑 密钥加载状态：
  ✅ DEEPSEEK_API_KEY
  ✅ FEISHU_APP_ID
  ✅ FEISHU_APP_SECRET
  ⚪ AI_PROVIDER  （可选，暂未配置）
  ⚪ ANTHROPIC_API_KEY  （可选，暂未配置）

🤖 当前 AI Provider: DEEPSEEK
   DEEPSEEK_API_KEY: ✅ 已加载


---
## ▶️ 第五步：选择运行模式

In [8]:
# @title 🕷️ 模式 A：运行爬虫（gcc_thinktank_scraper_v2.py）
import os, subprocess
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/gcc_scraper")

# @markdown ### 基础参数
MAX_PER_TANK = 40          # @param {type:"slider", min:5, max:200, step:5}
# @markdown 每个智库最多保留条数（默认 50，测试建议先用 10~20）
DAYS = 7                  # @param {type:"slider", min:1, max:90, step:1}
# @markdown 只收录近 N 天内发布的文章（0 = 不限）

# @markdown ### 功能开关
ENABLE_AI = True           # @param {type:"boolean"}
# @markdown 启用 AI 筛选、翻译、汇总（需要 DEEPSEEK_API_KEY）
NO_DEDUP = True           # @param {type:"boolean"}
# @markdown 禁用去重（每次全量处理，调试用）
DEBUG = False              # @param {type:"boolean"}
# @markdown 输出调试日志

# @markdown ### 高级参数（留空使用默认值）
COUNTRIES = ""             # @param {type:"string"}
# @markdown 只抓指定国家，多个用空格分隔（如 `US UK`），留空 = 全部
DEDUP_DAYS = 0             # @param {type:"slider", min:0, max:30, step:1}
# @markdown 去重时间窗口（天），0 = 关闭去重效果

# ── 自动检测并安装 Playwright Chromium ──────────────────────
chromium_path = "/root/.cache/ms-playwright/chromium_headless_shell-1208/chrome-headless-shell-linux64/chrome-headless-shell"
if not os.path.exists(chromium_path):
    print("🔧 检测到 Chromium 未安装，正在安装（约 1~2 分钟）...")
    r1 = subprocess.run("playwright install chromium", shell=True, capture_output=True, text=True)
    r2 = subprocess.run("playwright install-deps chromium", shell=True, capture_output=True, text=True)
    if os.path.exists(chromium_path):
        print("✅ Chromium 安装完成")
    else:
        print("⚠️  Chromium 安装可能失败，查看日志：")
        print(r1.stdout[-500:] or r1.stderr[-500:])
        print(r2.stdout[-500:] or r2.stderr[-500:])
else:
    print("✅ Chromium 已就绪，跳过安装")

# ── 构建命令 ────────────────────────────────────────────────
os.chdir(PROJECT_DIR)
entrypoint = f"{PROJECT_DIR}/gcc_thinktank_scraper_v2.py"
output_dir = f"{PROJECT_DIR}/output"
dedup_db   = f"{PROJECT_DIR}/data/gcc_dedup.db"

cmd_parts = [f"python {entrypoint}"]
cmd_parts.append(f"--max-per-tank {MAX_PER_TANK}")
cmd_parts.append(f"--days {DAYS}")
cmd_parts.append(f"--output-dir {output_dir}")
cmd_parts.append(f"--dedup-db {dedup_db}")
cmd_parts.append(f"--dedup-days {DEDUP_DAYS}")
if ENABLE_AI:         cmd_parts.append("--ai")
if NO_DEDUP:          cmd_parts.append("--no-dedup")
if DEBUG:             cmd_parts.append("--debug")
if COUNTRIES.strip(): cmd_parts.append(f"--countries {COUNTRIES.strip()}")

cmd = " ".join(cmd_parts)
print(f"\n🕷️ 启动爬虫")
print(f"   max_per_tank={MAX_PER_TANK} | days={DAYS} | ai={ENABLE_AI}")
print(f"   命令: {cmd}")
print("-" * 60)
result = subprocess.run(cmd, shell=True, text=True)
print("-" * 60)

# ── 统计输出文件 ─────────────────────────────────────────────
if os.path.exists(output_dir):
    files = sorted(os.listdir(output_dir))
    print(f"\n📁 output/ 目录共 {len(files)} 个文件")
    for f in files[-8:]:
        fpath = f"{output_dir}/{f}"
        size = os.path.getsize(fpath) / 1024
        print(f"  └── {f}  ({size:.1f} KB)")


🔧 检测到 Chromium 未安装，正在安装（约 1~2 分钟）...
✅ Chromium 安装完成

🕷️ 启动爬虫
   max_per_tank=40 | days=7 | ai=True
   命令: python /content/gcc_scraper/gcc_thinktank_scraper_v2.py --max-per-tank 40 --days 7 --output-dir /content/gcc_scraper/output --dedup-db /content/gcc_scraper/data/gcc_dedup.db --dedup-days 0 --ai --no-dedup
------------------------------------------------------------
------------------------------------------------------------

📁 output/ 目录共 28 个文件
  └── gcc_summary_20260423_1302.md  (109.1 KB)
  └── gcc_summary_20260423_1302.pdf  (162.5 KB)
  └── gcc_summary_20260424_1503.md  (31.4 KB)
  └── gcc_summary_20260424_1503.pdf  (46.8 KB)
  └── gcc_summary_20260427_0751.md  (32.5 KB)
  └── gcc_summary_20260427_0751.pdf  (49.1 KB)
  └── gcc_summary_20260427_1303.md  (31.3 KB)
  └── gcc_summary_20260427_1303.pdf  (48.9 KB)


In [10]:
# @title 🗃️ 模式 B：运行去重（dedup.py）
# @markdown 调用项目自带的 `dedup.py` 对数据库执行去重
import os
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/gcc_scraper")

SHOW_PREVIEW = True  # @param {type:"boolean"}
# @markdown **show_preview = True** 时额外展示去重后数据预览

import sys, os, sqlite3, pandas as pd
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

dedup_script = os.path.join(PROJECT_DIR, "dedup.py")
DB_PATH = os.path.join(PROJECT_DIR, "data", "gcc_dedup.db")

print("🗃️ 运行去重脚本: dedup.py")
print("-" * 55)
ret = os.system(f"python {dedup_script}")
print("-" * 55)

if ret == 0 and os.path.exists(DB_PATH) and SHOW_PREVIEW:
    conn = sqlite3.connect(DB_PATH)
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist()
    print(f"\n📊 去重后数据库概览 ({DB_PATH}):")
    for t in tables:
        count = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn)['n'][0]
        print(f"  └── 表 [{t}]: {count:,} 行")
    conn.close()
elif not os.path.exists(DB_PATH):
    print("\n⚠️  数据库尚不存在，请先运行模式 A 生成数据")

🗃️ 运行去重脚本: dedup.py
-------------------------------------------------------
-------------------------------------------------------


In [11]:
# @title 📤 模式 C：导出数据到 CSV / Excel
import os
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/gcc_scraper")

EXPORT_FORMAT = "Excel (.xlsx)"  # @param ["CSV (.csv)", "Excel (.xlsx)", "Both"]
TABLE_NAME = ""  # @param {type:"string"}
# @markdown 留空 TABLE_NAME 则导出所有表

import sqlite3, pandas as pd, os
from datetime import datetime

DB_PATH = os.path.join(PROJECT_DIR, "data", "gcc_dedup.db")
EXPORT_DIR = os.path.join(PROJECT_DIR, "data", "exports")
os.makedirs(EXPORT_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist()

target_tables = [TABLE_NAME] if TABLE_NAME else tables
exported = []

for table in target_tables:
    if table not in tables:
        print(f"⚠️  表 {table} 不存在")
        continue
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    print(f"📊 表 [{table}]: {len(df):,} 行 × {len(df.columns)} 列")

    if EXPORT_FORMAT in ["CSV (.csv)", "Both"]:
        path = f"{EXPORT_DIR}/{table}_{timestamp}.csv"
        df.to_csv(path, index=False, encoding='utf-8-sig')
        exported.append(path)
        print(f"  ✅ CSV → {path}")

    if EXPORT_FORMAT in ["Excel (.xlsx)", "Both"]:
        path = f"{EXPORT_DIR}/{table}_{timestamp}.xlsx"
        df.to_excel(path, index=False, engine='openpyxl')
        exported.append(path)
        print(f"  ✅ Excel → {path}")

conn.close()

# 提供下载链接
if exported:
    from google.colab import files
    print("\n📥 可选：下载到本地")
    DOWNLOAD_NOW = False  # 改为 True 立即下载
    if DOWNLOAD_NOW:
        for f in exported:
            files.download(f)

📊 表 [seen_articles]: 42 行 × 4 列
  ✅ Excel → /content/gcc_scraper/data/exports/seen_articles_20260424_132841.xlsx

📥 可选：下载到本地


In [ ]:
# @title 🚀 模式 D：合并简报到飞书文档（机器人空间版）
import os, re, time, requests
from google.colab import userdata
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/gcc_scraper")

# @markdown ### 推送配置
DOC_TITLE = "GCC智库研究动态简报汇编"  # @param {type:"string"}

# 👇 你和同事的 open_id（用于授权访问）
MY_OPEN_ID = "ou_70c12e982cb75faee8c86eba0d7ddfb1"  # @param {type:"string"}
COLLEAGUE_OPEN_IDS = [
    # "ou_同事的openid",
]

# ── 认证 ────────────────────────────────────────────────────
app_id     = re.sub(r'\s+', '', userdata.get("FEISHU_APP_ID"))
app_secret = re.sub(r'\s+', '', userdata.get("FEISHU_APP_SECRET"))
auth = requests.post(
    "https://open.feishu.cn/open-apis/auth/v3/tenant_access_token/internal",
    json={"app_id": app_id, "app_secret": app_secret}
).json()
if auth.get("code") != 0:
    raise RuntimeError(f"❌ 飞书认证失败: {auth}")
token = auth["tenant_access_token"]
headers = {"Authorization": f"Bearer {token}"}
print("✅ 飞书认证成功")

# ── 第一步：合并本地所有 MD 简报 ────────────────────────────
output_dir = os.path.join(PROJECT_DIR, "output")
md_files = sorted(
    [f for f in os.listdir(output_dir) if f.startswith("gcc_summary") and f.endswith(".md")],
    reverse=True
)
if not md_files:
    raise RuntimeError("❌ output/ 目录下没有找到简报 MD")

def shift_headings(content):
    result = []
    for line in content.split("\n"):
        m = re.match(r'^(#{1,5})\s', line)
        if m:
            result.append("#" + line)
        else:
            result.append(line)
    return "\n".join(result)

def filename_to_title(fname):
    m = re.search(r'(\d{4})(\d{2})(\d{2})_(\d{2})(\d{2})', fname)
    if m:
        y, mo, d, h, mi = m.groups()
        return f"{y}-{mo}-{d} {h}:{mi} 简报"
    return fname

merged = [f"# {DOC_TITLE}\n"]
merged.append(f"> 本文档自动汇总每日抓取的 GCC 智库简报，最新简报排在最前。共收录 {len(md_files)} 份简报。\n\n---\n")

for fname in md_files:
    with open(os.path.join(output_dir, fname), encoding="utf-8") as f:
        content = f.read()
    title = filename_to_title(fname)
    merged.append(f"\n# {title}\n")
    merged.append(shift_headings(content))
    merged.append("\n\n---\n")

merged_text = "\n".join(merged)
merged_path = "/tmp/gcc_merged.md"
with open(merged_path, "w", encoding="utf-8") as f:
    f.write(merged_text)

print(f"📝 合并完成：{len(md_files)} 份简报 | {len(merged_text):,} 字符")

# ── 第二步：上传 MD 到机器人空间 ────────────────────────────
file_size = os.path.getsize(merged_path)
with open(merged_path, "rb") as f:
    up = requests.post(
        "https://open.feishu.cn/open-apis/drive/v1/files/upload_all",
        headers={"Authorization": f"Bearer {token}"},
        data={
            "file_name": f"{DOC_TITLE}.md",
            "parent_type": "explorer",
            "parent_node": "",  # 留空，传到机器人默认空间
            "size": file_size,
        },
        files={"file": (f"{DOC_TITLE}.md", f, "text/markdown")}
    ).json()
if up.get("code") != 0:
    raise RuntimeError(f"上传失败: {up}")
file_token = up["data"]["file_token"]
print(f"⏫ 文件已上传")

# ── 第三步：导入为飞书文档（挂载到机器人自己的空间）─────────
imp = requests.post(
    "https://open.feishu.cn/open-apis/drive/v1/import_tasks",
    headers={**headers, "Content-Type": "application/json"},
    json={
        "file_extension": "md",
        "file_token": file_token,
        "type": "docx",
        "point": {
            "mount_type": 1,
            "mount_key": ""  # 👈 空字符串 = 挂载到根目录
        }
    }
).json()
if imp.get("code") != 0:
    raise RuntimeError(f"创建导入任务失败: {imp}")
ticket = imp["data"]["ticket"]

# ── 第四步：轮询等待导入完成 ────────────────────────────────
doc_url, doc_token = None, None
for i in range(30):
    time.sleep(3)
    s = requests.get(
        f"https://open.feishu.cn/open-apis/drive/v1/import_tasks/{ticket}",
        headers={"Authorization": f"Bearer {token}"}
    ).json()
    status = s.get("data", {}).get("result", {}).get("job_status")
    if status == 0:
        doc_url   = s["data"]["result"]["url"]
        doc_token = s["data"]["result"]["token"]
        print(f"✅ 文档导入成功")
        break
    elif status not in [1, 2, 4, 5, 6, 7]:  # 非中间状态
        err = s.get("data", {}).get("result", {}).get("job_error_msg", "未知错误")
        raise RuntimeError(f"❌ 导入失败 status={status}: {err}")

if not doc_url:
    raise RuntimeError("❌ 导入超时")

# ── 第五步：自动授权给你和同事 (full_access) ────────────────
def add_doc_member(doc_token, open_id):
    return requests.post(
        f"https://open.feishu.cn/open-apis/drive/v1/permissions/{doc_token}/members",
        headers={**headers, "Content-Type": "application/json"},
        params={"type": "docx", "need_notification": "false"},
        json={
            "member_type": "openid",
            "member_id": open_id,
            "perm": "full_access",
            "type": "user"
        }
    ).json()

if MY_OPEN_ID:
    r = add_doc_member(doc_token, MY_OPEN_ID)
    if r.get("code") == 0:
        print(f"👤 已授权给你 (full_access)")
    else:
        print(f"⚠️ 授权你失败: {r}")

for uid in COLLEAGUE_OPEN_IDS:
    r = add_doc_member(doc_token, uid)
    if r.get("code") == 0:
        print(f"👥 已授权同事 {uid[:10]}...")
    else:
        print(f"⚠️ 授权同事失败: {r}")

# ── 第六步：开启链接分享（组织内可查看）─────────────────────
perm = requests.patch(
    f"https://open.feishu.cn/open-apis/drive/v1/permissions/{doc_token}/public",
    headers={**headers, "Content-Type": "application/json"},
    params={"type": "docx"},
    json={"link_share_entity": "tenant_readable"}
).json()
if perm.get("code") == 0:
    print(f"🔓 组织内任何人通过链接可查看")

# ── 汇总 ────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"🎉 简报汇编更新完成！")
print(f"📄 共收录: {len(md_files)} 份简报")
print(f"📎 文档链接: {doc_url}")
print(f"\n💡 直接把上面的链接发给同事即可，无需文件夹")

✅ 飞书认证成功


In [17]:
import requests, re
from google.colab import userdata

# 认证（如果当前 cell 已经有 token 就跳过这段）
app_id     = re.sub(r'\s+', '', userdata.get("FEISHU_APP_ID"))
app_secret = re.sub(r'\s+', '', userdata.get("FEISHU_APP_SECRET"))
auth = requests.post(
    "https://open.feishu.cn/open-apis/auth/v3/tenant_access_token/internal",
    json={"app_id": app_id, "app_secret": app_secret}
).json()
token = auth["tenant_access_token"]

# 通过手机号查 open_id
def get_open_id(mobile):
    r = requests.post(
        "https://open.feishu.cn/open-apis/contact/v3/users/batch_get_id",
        headers={"Authorization": f"Bearer {token}"},
        params={"user_id_type": "open_id"},
        json={"mobiles": [mobile]}  # 国际格式: +86138xxxxxxxx
    ).json()
    print(r)
    return r

# 替换为同事的手机号（含国家码 +86）
get_open_id("+86138xxxxxxxx")

{'code': 0, 'data': {'user_list': [{'mobile': '+86138xxxxxxxx'}]}, 'msg': 'success'}


{'code': 0,
 'data': {'user_list': [{'mobile': '+86138xxxxxxxx'}]},
 'msg': 'success'}

In [36]:
# @title 📄 模式 E：全文转 PDF（fulltext_to_pdf.py）
# @markdown 将抓取的全文内容转换为 PDF，输出到 `output_fulltext/` 目录
import os
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/gcc_scraper")

PDF_LIMIT = 20  # @param {type:"slider", min:1, max:200, step:1}
# @markdown `PDF_LIMIT`：本次最多处理的文章数量（避免运行时间过长）
EXTRA_PDF_ARGS = ""  # @param {type:"string"}

import os
os.chdir(PROJECT_DIR)

script = os.path.join(PROJECT_DIR, "fulltext_to_pdf.py")
cmd_parts = [f"python {script}", f"--limit {PDF_LIMIT}"]
if EXTRA_PDF_ARGS: cmd_parts.append(EXTRA_PDF_ARGS)
cmd = " ".join(cmd_parts)

print(f"📄 全文转 PDF | limit={PDF_LIMIT}")
print(f"   命令: {cmd}")
print("-" * 55)
os.system(cmd)
print("-" * 55)

# 统计输出
pdf_dir = os.path.join(PROJECT_DIR, "output_fulltext")
if os.path.exists(pdf_dir):
    pdfs = [f for f in os.listdir(pdf_dir) if f.endswith(".pdf")]
    total_mb = sum(os.path.getsize(f"{pdf_dir}/{f}") for f in pdfs) / 1024 / 1024
    print(f"\n📁 output_fulltext/ 共 {len(pdfs)} 个 PDF，总大小 {total_mb:.1f} MB")
    for f in sorted(pdfs)[-5:]:
        size = os.path.getsize(f"{pdf_dir}/{f}") / 1024
        print(f"  └── {f}  ({size:.1f} KB)")

📄 全文转 PDF | limit=20
   命令: python /content/gcc_scraper/fulltext_to_pdf.py --limit 20
-------------------------------------------------------
-------------------------------------------------------

📁 output_fulltext/ 共 2 个 PDF，总大小 0.5 MB
  └── ajcs_20260416_1051.pdf  (259.3 KB)
  └── ajcs_20260416_1942.pdf  (259.3 KB)


---
## 👥 第六步：协作管理工具

In [2]:
# @title 📤 同步 Colab 笔记本 & 推送到 GitHub
import os, shutil, subprocess, re
from google.colab import userdata
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/gcc_scraper")

# @markdown ### 提交配置
COMMIT_MESSAGE = "feishu update"  # @param {type:"string"}
GIT_USER_NAME  = "Jungle225TT"  # @param {type:"string"}
GIT_USER_EMAIL = "jungle20010225@gmail.com"  # @param {type:"string"}
CONFIRM_PUSH = True  # @param {type:"boolean"}
# @markdown ⚠️ 填写 commit message 后勾选 CONFIRM_PUSH 再运行

# 检查项目目录是否存在
if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(f"❌ 项目目录不存在: {PROJECT_DIR}\n   请先运行第一步拉取代码！")
os.chdir(PROJECT_DIR)

def run(cmd, show=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=PROJECT_DIR)
    if show:
        out = (r.stdout + r.stderr).strip()
        if out: print(out)
    return r.returncode

# ── 1. 配置 git 身份和认证 ──────────────────────────────────
run(f'git config user.name "{GIT_USER_NAME}"', show=False)
run(f'git config user.email "{GIT_USER_EMAIL}"', show=False)

try:
    token_val = re.sub(r'\s+', '', userdata.get("GITHUB_TOKEN"))
    run(f'git remote set-url origin https://{token_val}@github.com/Jungle225TT/GCC-.git', show=False)
except Exception:
    print("⚠️  未找到 GITHUB_TOKEN，推送可能失败")

# ── 2. 同步 Colab 笔记本 ────────────────────────────────────
possible_paths = [
    "/content/drive/MyDrive/Colab Notebooks/GCC_Scraper_Colab.ipynb",
    "/content/drive/My Drive/Colab Notebooks/GCC_Scraper_Colab.ipynb",
]
for p in possible_paths:
    if os.path.exists(p):
        target = os.path.join(PROJECT_DIR, "notebooks", "GCC_Scraper_Colab.ipynb")
        os.makedirs(os.path.dirname(target), exist_ok=True)
        shutil.copy2(p, target)
        print(f"📔 Colab 笔记本已同步")
        break

# ── 3. 显示待提交改动 ───────────────────────────────────────
print("\n📋 待提交改动：")
run("git status --short")

# ── 4. 提交并推送 ───────────────────────────────────────────
if not CONFIRM_PUSH:
    print("\n⏸️  检查无误后勾选 CONFIRM_PUSH 再运行")
else:
    run("git add -A")
    commit_ret = subprocess.run(
        f'git commit -m "{COMMIT_MESSAGE}"',
        shell=True, capture_output=True, text=True, cwd=PROJECT_DIR
    )
    output = commit_ret.stdout + commit_ret.stderr
    print(output)

    if "nothing to commit" not in output:
        push_ret = run("git push origin main")
        if push_ret == 0:
            print("\n✅ 推送成功！https://github.com/Jungle225TT/GCC-")
        else:
            print("\n❌ 推送失败，请检查 GITHUB_TOKEN")

FileNotFoundError: [Errno 2] No such file or directory: '/content/gcc_scraper'

In [ ]:
# @title 🔄 同步代码更新（拉取最新代码 + 保留本地数据）
# @markdown 同事推送代码后，运行此格获取最新版本
import os, subprocess
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/gcc_scraper")

BRANCH = "main"  # @param ["main", "dev"]
STASH_LOCAL = True  # @param {type:"boolean"}
# @markdown **stash_local = True**：暂存本地未提交改动，避免冲突

os.chdir(PROJECT_DIR)

def git(cmd):
    r = subprocess.run(f"git {cmd}", shell=True, capture_output=True, text=True, cwd=PROJECT_DIR)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(f"  {out}")
    return r.returncode

print("🔄 同步代码...")

if STASH_LOCAL:
    print("📦 暂存本地改动")
    git("stash")

print(f"⬇️  拉取 origin/{BRANCH}")
code = git(f"pull origin {BRANCH}")

if STASH_LOCAL:
    print("📤 恢复本地改动")
    git("stash pop")

print(f"\n{'✅ 同步成功' if code == 0 else '⚠️  同步遇到冲突，请手动处理'}")
git("log --oneline -5")

In [44]:
# @title 📊 项目状态总览（随时运行查看）
import os
PROJECT_DIR = globals().get("PROJECT_DIR", "/content/gcc_scraper")

import os, sqlite3, subprocess
from datetime import datetime

print("=" * 55)
print("  📊 GCC Scraper — 项目状态总览")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 55)

# Git 状态
r = subprocess.run("git log --oneline -3", shell=True, capture_output=True, text=True, cwd=PROJECT_DIR)
print("\n📌 最近提交：")
for line in r.stdout.strip().split("\n"):
    print(f"  {line}")

# Drive 挂载状态
drive_ok = os.path.ismount("/content/drive")
print(f"\n💾 Google Drive: {'✅ 已挂载' if drive_ok else '❌ 未挂载'}")

# 各目录状态
CHECK_DIRS = [
    ("data",            "数据库目录"),
    ("output",          "爬虫输出"),
    ("output_fulltext", "全文 PDF"),
]
print("\n📁 目录状态：")
for dname, label in CHECK_DIRS:
    dpath = os.path.join(PROJECT_DIR, dname)
    is_link = os.path.islink(dpath)
    exists = os.path.exists(dpath)
    if exists:
        count = len(os.listdir(dpath))
        link_mark = "🔗" if is_link else "📂"
        print(f"  {link_mark} {dname}/  [{label}]  {count} 个文件")
    else:
        print(f"  ❌ {dname}/  [{label}]  不存在")

# 数据库状态
DB_PATH = os.path.join(PROJECT_DIR, "data", "gcc_dedup.db")
if os.path.exists(DB_PATH):
    size_mb = os.path.getsize(DB_PATH) / 1024 / 1024
    conn = sqlite3.connect(DB_PATH)
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    print(f"\n🗃️  数据库: {size_mb:.2f} MB")
    for (t,) in tables:
        count = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        print(f"  └── {t}: {count:,} 行")
    conn.close()
else:
    print("\n🗃️  数据库: 尚未创建")

# 密钥状态
keys = ["FEISHU_APP_ID", "FEISHU_APP_SECRET", "SCRAPER_API_KEY"]
print("\n🔑 密钥：")
for k in keys:
    status = "✅" if os.environ.get(k) else "⚠️ "
    print(f"  {status} {k}")

print("\n" + "=" * 55)

  📊 GCC Scraper — 项目状态总览
  2026-04-24 14:49:55

📌 最近提交：
  5f01425 feat: 更新模式D为合并简报到单文档方案；将 output 目录移出 git 跟踪
  cbdc867 structure
  98d9ff8 filter_rules

💾 Google Drive: ✅ 已挂载

📁 目录状态：
  🔗 data/  [数据库目录]  2 个文件
  🔗 output/  [爬虫输出]  16 个文件
  🔗 output_fulltext/  [全文 PDF]  4 个文件

🗃️  数据库: 0.03 MB
  └── seen_articles: 42 行

🔑 密钥：
  ⚠️  FEISHU_APP_ID
  ⚠️  FEISHU_APP_SECRET
  ⚠️  SCRAPER_API_KEY



---
## 📌 协作指南

### 同事首次使用
1. 获取共享 Drive 文件夹路径（如 `MyDrive/gcc_project`）
2. 打开此笔记本（File → Save a copy in Drive 保存自己的副本）
3. 修改第二步中的 `DRIVE_ROOT` 为共享路径
4. 在左侧 🔑 Secrets 中添加 API 密钥
5. 按顺序运行第 1–4 步完成初始化

### 日常使用流程
```
每次打开 Colab 后（环境会重置）：
  ① 运行「第一步」→ 拉取最新代码
  ② 运行「第二步」→ 重新挂载 Drive（必须）
  ③ 运行「第四步」→ 重新加载密钥（必须）
  ④ 直接跑需要的模式 A / B / C / D / E
```

### 真实目录结构
```
gcc_scraper/                         ← GitHub 代码（每次从 GitHub 拉取）
├── gcc_thinktank_scraper_v2.py      ← 爬虫主入口（模式 A）
├── dedup.py                         ← 去重脚本（模式 B）
├── feishu_sync.py                   ← 飞书同步（模式 D）
├── fulltext_to_pdf.py               ← 全文转 PDF（模式 E）
├── ai_client.py                     ← AI 客户端
├── filter_rules.yaml                ← 过滤规则配置
├── scoring_test.py                  ← 评分测试
├── requirements.txt
├── data/           → 🔗 Drive/gcc_project/data/           （数据库）
├── output/         → 🔗 Drive/gcc_project/output/         （抓取结果）
└── output_fulltext/→ 🔗 Drive/gcc_project/output_fulltext/（全文 PDF）
```

### ⚠️ 注意事项
- Colab 运行环境每次重启后，**必须重新运行第二步挂载 Drive**，否则 output/ 等目录无法写入
- `gcc_thinktank_scraper_v2.py` 如有命令行参数，在「额外参数」栏填入
- 多人同时运行爬虫写同一数据库时，建议错峰操作或在 `dedup.py` 中启用 WAL 模式
- 飞书同步前请确认 `FEISHU_APP_TOKEN` 和 `FEISHU_TABLE_ID` 正确